# CSoNet 2026 Camera-Ready Experiments — PhoBERT, 5 Seeds (v3)

This version fixes the Kaggle warnings seen in v2 and makes the smoke test easier to diagnose.

### Changes in v3
- suppresses the harmless Hugging Face unauthenticated warning;
- patches the pandas `groupby.apply` FutureWarning;
- uses the current `torch.amp` API instead of deprecated CUDA AMP calls;
- runs the smoke test with `num_workers=0` and batch size 8;
- removes stale smoke-test outputs before each smoke run;
- prints a concise error tail if a subprocess really fails.


In [ ]:
# 1) SAFE environment check — do NOT upgrade Kaggle's scientific stack
import sys, importlib, subprocess, os, warnings
from importlib.metadata import version, PackageNotFoundError

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", category=FutureWarning)

required = {
    "torch": None,
    "numpy": None,
    "pandas": None,
    "scipy": None,
    "sklearn": None,
    "transformers": "transformers>=4.40,<5",
    "rdflib": "rdflib",
    "openpyxl": "openpyxl",
    "sentencepiece": "sentencepiece",
}
module_to_pip = {"sklearn": "scikit-learn"}

missing = []
for module, install_spec in required.items():
    try:
        importlib.import_module(module)
    except Exception:
        missing.append(install_spec or module_to_pip.get(module, module))

if missing:
    print("Installing only missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available; nothing is upgraded.")

for pkg in ["torch", "numpy", "pandas", "scipy", "scikit-learn", "transformers", "rdflib", "openpyxl"]:
    try:
        print(f"{pkg:14s}: {version(pkg)}")
    except PackageNotFoundError:
        pass

print("\nEnvironment check completed.")


In [ ]:
# 2) Locate package and create a PATCHED runner in /kaggle/working
from pathlib import Path
import zipfile, os, shutil, re, py_compile

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()

def choose_script(candidates):
    if not candidates:
        return None
    preferred = [p for p in candidates if "EmotionOntology_CameraReady_Experiments" in str(p)]
    return preferred[0] if preferred else candidates[0]

script_candidates = list(INPUT_ROOT.rglob("camera_ready_experiments.py")) if INPUT_ROOT.exists() else []
ORIGINAL_SCRIPT = choose_script(script_candidates)

if ORIGINAL_SCRIPT is None:
    zip_candidates = list(INPUT_ROOT.rglob("*.zip")) if INPUT_ROOT.exists() else []
    package_zip = None
    for z in zip_candidates:
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("camera_ready_experiments.py") for n in zf.namelist()):
                    package_zip = z
                    break
        except Exception:
            pass
    if package_zip is None:
        raise FileNotFoundError(
            "Could not find camera_ready_experiments.py or a ZIP containing it under /kaggle/input."
        )
    extract_dir = WORK / "camera_ready_package"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_zip) as zf:
        zf.extractall(extract_dir)
    ORIGINAL_SCRIPT = choose_script(list(extract_dir.rglob("camera_ready_experiments.py")))

PACKAGE_ROOT = ORIGINAL_SCRIPT.parent
INPUT_DIR = PACKAGE_ROOT / "input"

if not (INPUT_DIR / "synthetic_train.csv").exists():
    matched = []
    for tr in (list(INPUT_ROOT.rglob("synthetic_train.csv")) if INPUT_ROOT.exists() else []):
        if (tr.parent / "synthetic_val.csv").exists():
            matched.append(tr.parent)
    if not matched:
        raise FileNotFoundError("Could not locate synthetic_train.csv + synthetic_val.csv.")
    INPUT_DIR = matched[0]

text = ORIGINAL_SCRIPT.read_text(encoding="utf-8")

# Add quiet-mode settings inside the subprocess script.
if 'warnings.filterwarnings("ignore", category=FutureWarning)' not in text:
    text = text.replace(
        "import unicodedata\n",
        'import unicodedata\nimport warnings\n'
        'warnings.filterwarnings("ignore", category=FutureWarning)\n'
        'os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n'
        'os.environ.setdefault("HF_HUB_VERBOSITY", "error")\n'
    )

# Replace the deprecated pandas groupby.apply subsampling.
old_sampling = '''    if max_train_samples and len(tr) > max_train_samples:
        tr = tr.groupby("label", group_keys=False).apply(
            lambda g: g.sample(max(1, round(max_train_samples * len(g) / len(tr))), random_state=seed)
        ).reset_index(drop=True).head(max_train_samples)
    if max_dev_samples and len(dev) > max_dev_samples:
        dev = dev.groupby("label", group_keys=False).apply(
            lambda g: g.sample(max(1, round(max_dev_samples * len(g) / len(dev))), random_state=seed)
        ).reset_index(drop=True).head(max_dev_samples)
'''

new_sampling = '''    def _stratified_cap(frame: pd.DataFrame, cap: Optional[int], rs: int) -> pd.DataFrame:
        if cap is None or cap <= 0 or len(frame) <= cap:
            return frame.reset_index(drop=True)
        pieces = []
        n_total = len(frame)
        for _, g in frame.groupby("label", sort=False):
            take = max(1, round(cap * len(g) / n_total))
            pieces.append(g.sample(n=min(take, len(g)), random_state=rs))
        out = pd.concat(pieces, axis=0)
        if len(out) > cap:
            out = out.sample(n=cap, random_state=rs)
        elif len(out) < cap:
            remaining = frame.drop(index=out.index, errors="ignore")
            if len(remaining):
                add_n = min(cap - len(out), len(remaining))
                out = pd.concat([out, remaining.sample(n=add_n, random_state=rs)], axis=0)
        return out.sample(frac=1.0, random_state=rs).reset_index(drop=True)

    tr = _stratified_cap(tr, max_train_samples, seed)
    dev = _stratified_cap(dev, max_dev_samples, seed)
'''

if old_sampling in text:
    text = text.replace(old_sampling, new_sampling)
else:
    print("NOTE: sampling block was already patched or differs from v2.")

# Modern PyTorch AMP API.
text = text.replace(
    "from torch.cuda.amp import autocast, GradScaler",
    "from torch.amp import autocast, GradScaler"
)
text = text.replace(
    "scaler = GradScaler(enabled=use_amp)",
    'scaler = GradScaler("cuda", enabled=use_amp)'
)
text = text.replace(
    "with autocast(enabled=use_amp):",
    'with autocast(device_type=device.type, enabled=use_amp):'
)

PATCHED_SCRIPT = WORK / "camera_ready_experiments_v3.py"
PATCHED_SCRIPT.write_text(text, encoding="utf-8")
SCRIPT = PATCHED_SCRIPT
py_compile.compile(str(SCRIPT), doraise=True)

OUTPUT_DIR = WORK / "csonet_camera_ready_outputs"
SMOKE_DIR = WORK / "csonet_smoke_outputs"

print("ORIGINAL_SCRIPT =", ORIGINAL_SCRIPT)
print("PATCHED_SCRIPT  =", SCRIPT)
print("INPUT_DIR       =", INPUT_DIR)
print("OUTPUT_DIR      =", OUTPUT_DIR)

for name in ["synthetic_train.csv", "synthetic_val.csv"]:
    p = INPUT_DIR / name
    assert p.exists(), f"Missing required file: {p}"
    print(f"OK: {name} ({p.stat().st_size/1024:.1f} KiB)")

print("\nPatched runner created and syntax-checked successfully.")


In [ ]:
# 3) GPU + PhoBERT availability check
import torch, os
from huggingface_hub import logging as hf_logging
hf_logging.set_verbosity_error()

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError(
        "GPU is not enabled. In Kaggle Settings select a GPU accelerator, restart the session, "
        "and rerun from Cell 1."
    )

from transformers import AutoTokenizer
MODEL_NAME = "vinai/phobert-base-v2"
try:
    _tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("PhoBERT tokenizer is available:", MODEL_NAME)
    print("Cell 3 OK. No HF token is required for this public model.")
except Exception:
    print("\nCould not load PhoBERT tokenizer. If this is an Internet/cache error, enable Internet and rerun.")
    raise


In [ ]:
# 4) PREFLIGHT — validate benchmark and export support statistics
import subprocess, sys, pandas as pd, os
from collections import deque

def run_clean(cmd):
    env = os.environ.copy()
    env["HF_HUB_VERBOSITY"] = "error"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTHONWARNINGS"] = "ignore::FutureWarning"
    print("Running:", " ".join(map(str, cmd)))
    tail = deque(maxlen=80)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"Subprocess failed with exit code {rc}.\n\nLast output lines:\n{''.join(tail)}"
        )
    return rc

cmd = [
    sys.executable, str(SCRIPT),
    "--input_dir", str(INPUT_DIR),
    "--output_dir", str(OUTPUT_DIR),
    "--preflight_only",
]
run_clean(cmd)

print("\nDataset statistics:")
display(pd.read_csv(OUTPUT_DIR / "dataset_statistics.csv"))

for optional_name, title in [
    ("ontology_kg_statistics.csv", "Ontology/KG statistics"),
    ("human_annotation_iaa.csv", "Human annotation IAA"),
    ("sparql_representative_results.csv", "Representative SPARQL results"),
]:
    p = OUTPUT_DIR / optional_name
    if p.exists():
        print(f"\n{title}:")
        display(pd.read_csv(p))


## 5) Smoke test — run this before the 5-seed experiment

This uses **seed 1**, **1 epoch**, and a small training/development subset.  
It is only a technical check; **do not report its metrics in the paper**.


In [ ]:
# 5) SMOKE TEST — deliberately small and clean
import sys, pandas as pd, shutil

# Remove stale smoke-test outputs so success always refers to THIS run.
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

cmd = [
    sys.executable, str(SCRIPT),
    "--input_dir", str(INPUT_DIR),
    "--output_dir", str(SMOKE_DIR),
    "--run_phobert",
    "--seeds", "1",
    "--epochs", "1",
    "--batch_size", "8",
    "--num_workers", "0",
    "--max_train_samples", "600",
    "--max_dev_samples", "240",
    "--force",
]
run_clean(cmd)

smoke_metrics = SMOKE_DIR / "all_seed_metrics.csv"
if not smoke_metrics.exists():
    raise FileNotFoundError("Smoke test completed without all_seed_metrics.csv.")

smoke_df = pd.read_csv(smoke_metrics)
display(smoke_df)

expected = {"PhoBERT-Text", "PhoBERT+Topic", "PhoBERT+Ontology"}
found = set(smoke_df["Model"].astype(str))
missing_models = expected - found
if missing_models:
    raise RuntimeError(f"Smoke test output is incomplete. Missing models: {sorted(missing_models)}")

print("\nSMOKE TEST COMPLETED SUCCESSFULLY.")
print("You may now run Cell 6 (full five-seed experiment).")


## 6) Full camera-ready run — 5 seeds

Run this cell only after the smoke test succeeds.

Protocol:
- PhoBERT: `vinai/phobert-base-v2`
- seeds: **1, 2, 3, 4, 5**
- max sequence length: **128**
- batch size: **16**
- epochs: **5** maximum, with development-set early stopping
- optimizer: **AdamW**
- final evaluation: fixed `synthetic_val.csv`

If Kaggle disconnects after one or more seeds have completed, rerun this cell. The experiment script is designed to reuse completed seed outputs.


In [ ]:
# 6) FULL FIVE-SEED RUN
import sys

cmd = [
    sys.executable, str(SCRIPT),
    "--input_dir", str(INPUT_DIR),
    "--output_dir", str(OUTPUT_DIR),
    "--run_phobert",
    "--seeds", "1", "2", "3", "4", "5",
    "--epochs", "5",
    "--batch_size", "16",
    "--num_workers", "0",
]
run_clean(cmd)

print("\nFULL RUN COMMAND FINISHED.")


In [ ]:
# 7) Inspect camera-ready outputs
import pandas as pd

required_outputs = [
    "all_seed_metrics.csv",
    "summary_mean_std.csv",
    "paper_ready_table.csv",
]
for name in required_outputs:
    p = OUTPUT_DIR / name
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. The full five-seed run has not completed successfully yet.")

all_seed = pd.read_csv(OUTPUT_DIR / "all_seed_metrics.csv")
summary = pd.read_csv(OUTPUT_DIR / "summary_mean_std.csv")
paper = pd.read_csv(OUTPUT_DIR / "paper_ready_table.csv")

print("Per-seed metrics:")
display(all_seed)

print("\nMean ± std (numeric):")
display(summary)

print("\nPaper-ready table:")
display(paper)

notes = OUTPUT_DIR / "camera_ready_replacement_notes.md"
if notes.exists():
    print("\nCamera-ready replacement notes:\n")
    print(notes.read_text(encoding="utf-8"))


In [ ]:
# 8) Package outputs for download
import shutil
from pathlib import Path

zip_base = Path("/kaggle/working/camera_ready_outputs") if Path("/kaggle/working").exists() else Path.cwd() / "camera_ready_outputs"
zip_file = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

print("Created:", zip_file)
print("Download this ZIP from the Kaggle Output/Files panel and send it back for manuscript v5 updating.")
